In [1]:
import pandas as pd

file_path = r"E:\apa_qtl_GTEx\GTEx_apa_qtl_brain\Brain_Amygdala.v10.cis_apaqtl.allpairs.chr1.parquet"

# Force pandas to use fastparquet instead of pyarrow
df = pd.read_parquet(file_path, engine='fastparquet')

print(df.iloc[:5, :])
#print(df.tail(5))

                           phenotype_id          variant_id  tss_distance  \
0  ENSG00000225972.1_chr1:629062-629433  chr1_13550_G_A_b38       -615512   
1  ENSG00000225972.1_chr1:629062-629433  chr1_14671_G_C_b38       -614391   
2  ENSG00000225972.1_chr1:629062-629433  chr1_16841_G_T_b38       -612221   
3  ENSG00000225972.1_chr1:629062-629433  chr1_16856_A_G_b38       -612206   
4  ENSG00000225972.1_chr1:629062-629433  chr1_17005_A_G_b38       -612057   

         af  ma_samples  ma_count  pval_nominal     slope  slope_se  
0  0.013889           5         5      0.735234 -0.172060  0.507882  
1  0.002778           1         1      0.179244  1.466603  1.087025  
2  0.036111          13        13      0.081389  0.534934  0.304962  
3  0.005556           2         2      0.049751  1.511794  0.764473  
4  0.013889           5         5      0.944614  0.034218  0.491741  


In [2]:
# Open .txt.gz file

import gzip

# Replace with the actual path to your file
file_path = r"D:\Project_CompBio\Summer_2026\QTL_Project\apaQTL and ipaQTL\GTEx_apaQTL\GTEx_Analysis_v11_apaQTL\Adipose_Subcutaneous.v11.apaGenes.txt.gz"

# 'rt' stands for "read text" (handles decoding the bytes into strings)
with gzip.open(file_path, 'rt', encoding='utf-8') as file:
    for i in range(5):
        line = file.readline()
        
        # Stop if the file is empty or has fewer than 5 lines
        if not line:
            break
            
        # .strip() removes the extra newline character at the end of each row
        print(line.strip())

phenotype_id	gene_id	gene_name	biotype	gene_chr	gene_start	gene_end	strand	num_var	beta_shape1	beta_shape2	true_df	pval_true_df	variant_id	tss_distance	chr	variant_pos	ref	alt	num_alt_per_site	rs_id_dbSNP157_GRCh38p14	ma_samples	ma_count	af	pval_nominal	slope	slope_se	pval_perm	pval_beta	group_size	qval	pval_nominal_threshold
ENSG00000308579.1_chr1:134914-135140	ENSG00000308579.1	ENSG00000308579	lncRNA	chr1	134914	136246	-	1857	1.03586	325.328	553.657	0.00127238	chr1_1103416_G_A_b38	967170	chr1	1103416	G	A	1	rs115075475	19	19	0.0133615	0.000525905	-0.843383	0.242011	0.322868	0.322317	1	0.344368	8.48126e-05
ENSG00000225972.1_chr1:629062-629433	ENSG00000225972.1	MTND1P23	unprocessed_pseudogene	chr1	629062	629433	+	4326	1.03257	705.201	570.337	4.04653e-05	chr1_1285621_A_G_b38	656559	chr1	1285621	A	G	1	rs72896210	39	45	0.0316456	1.34974e-05	0.64373	0.146771	0.0241976	0.0246999	1	0.0520604	3.86212e-05
ENSG00000225630.1_chr1:629640-630683	ENSG00000225630.1	MTND2P28	unprocessed_pseudogene	chr

In [ ]:
# check the 28-29 Mb region on the X chromosome for apaQTLs

import polars as pl

# 1. Define the file path
file_path = r'E:\apa_qtl_GTEx\GTEx_apa_qtl_brain\Brain_Cortex.v10.cis_apaqtl.allpairs.chrX.parquet'

print(f"Scanning for variants between 28Mb and 29Mb...")

# 2. Open a LAZY scan (does not load the massive file into RAM)
lf = pl.scan_parquet(file_path)

# 3. Create a temporary column to extract the position, then filter it
# variant_id format: chrX_28500000_A_C_b38 -> split by '_' -> index 1 is the number
query = (
    lf
    .with_columns(
        pl.col("variant_id").str.split("_").list.get(1).cast(pl.Int64).alias("extracted_pos")
    )
    .filter(
        (pl.col("extracted_pos") >= 28_000_000) & 
        (pl.col("extracted_pos") <= 29_000_000)
    )
    .drop("extracted_pos") # Clean up the temporary column before showing the results
)

# 4. Execute the query and pull ONLY the matching rows into memory
try:
    df_filtered = query.collect()
    
    # 5. Check the results
    if len(df_filtered) == 0:
        print("Result: 0 rows found. This genomic region is completely empty in this dataset.")
    else:
        print(f"Success! Found {len(df_filtered)} rows.")
        # Print the first 20 rows to the console so you can inspect them
        print(df_filtered.head(20))
        
        # Optional: Save it to a CSV if you want to look at it in Excel
        # df_filtered.write_csv("chrX_manual_check.tsv", separator='\t')
        
except Exception as e:
    print(f"An error occurred while scanning: {e}")

Scanning for variants between 28Mb and 29Mb...
Result: 0 rows found. This genomic region is completely empty in this dataset.


In [1]:
!pip install polars pyarrow

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/833.4 kB ? eta -:--:--
   ---------------------------------------- 833.4/833.4 kB 7.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/52.0 MB ? eta -:--:--
   - -------------------------------------- 2.4/52.0 MB 11.7 MB/s eta 0:00:05
   --- ------------------------------------ 5.0/52.0 MB 11.9 MB/s eta 0:00:04
   ----- ---------------------------------- 7.6/52.0 MB 12.0 MB/s eta 0:00:04
   ------- -------------------------------- 10.2/52.0 MB 11.9 MB/s eta 0:00:04
   --------- ------------------------------ 12.8/52.0 MB 12.0 MB/s eta 0:00:04
   ----------- ---------------------------- 15.2/52.0 MB 12.0 MB/s eta 0:00:04
   ------------- -------------------------- 17.8/52.0 MB 11.9 MB/s eta 0:00:03
   --------------- ------------------------ 20.4/52.0 MB 12.0 MB/s eta 0:00:03
   ----------------- ---------------------- 23.3/52.0 MB 12.0 MB/s eta 0


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# Obtain the apaQTLs for each tissue in the 2Mb region around each significant GWAS SNP

import polars as pl
import os
import glob
import re
from functools import reduce
import operator

# --- Configuration ---
TSV_FILE_PATH = r'D:\Project_CompBio\Summer_2026\QTL_Project\Sex-stratified GWAS\Major Depressive Disorder\male_sig_gwas_snps_hg38.tsv'
PARQUET_FOLDER = r'E:\apa_qtl_GTEx\GTEx_apa_qtl_brain'
WINDOW_SIZE = 1_000_000

def extract_apaqtl_windows():
    print("Loading target SNPs...")
    
    # 1. Load the target SNPs
    try:
        df_snps = pl.read_csv(TSV_FILE_PATH, separator='\t')
    except Exception as e:
        print(f"Error loading TSV: {e}")
        return

    # 2. Get unique chromosomes to process them one by one
    chromosomes = df_snps.select("chromosome").unique().to_series().to_list()
    
    for chrom in chromosomes:
        # Normalize chromosome string (ensure it doesn't have 'chr' prefix to match filename)
        chrom_str = str(chrom).replace("chr", "")
        
        # Get all SNPs on this specific chromosome
        chrom_snps = df_snps.filter(pl.col("chromosome") == chrom)
        
        # Build the exact search pattern for the parquet files
        # e.g., Brain_*.v10.cis_apaqtl.allpairs.chr1.parquet
        pattern = os.path.join(PARQUET_FOLDER, f"Brain_*.v10.cis_apaqtl.allpairs.chr{chrom_str}.parquet")
        parquet_files = glob.glob(pattern)
        
        if not parquet_files:
            print(f"Warning: No parquet files found for chromosome {chrom_str}.")
            continue
            
        print(f"\n--- Processing Chromosome {chrom_str} ({len(chrom_snps)} SNPs) ---")
        
        # Prepare the window ranges for all SNPs on this chromosome
        snp_windows = []
        for row in chrom_snps.iter_rows(named=True):
            rs_id = row['rs_id']
            bp = row['base_pair_location']
            lower_bound = bp - WINDOW_SIZE
            upper_bound = bp + WINDOW_SIZE
            snp_windows.append((rs_id, lower_bound, upper_bound))
        
        # 3. Iterate through each tissue parquet file for this chromosome
        for pq_file in parquet_files:
            # Extract tissue name using regex
            match = re.search(r"Brain_(.+?)\.v10", os.path.basename(pq_file))
            if not match:
                continue
            tissue = match.group(1)
            print(f"Scanning Brain_{tissue}...")

            # Open a LAZY scan of the massive parquet file (does not load into RAM yet)
            lf = pl.scan_parquet(pq_file)
            
            # Create a dynamic column that extracts the position from the variant_id string
            # variant_id format: chr1_10500_A_T_b38 -> splitting by '_' and taking index 1
            lf = lf.with_columns(
                pl.col("variant_id").str.split("_").list.get(1).cast(pl.Int64).alias("extracted_pos")
            )
            
            # Build a combined OR filter for ALL SNPs on this chromosome
            # This ensures we only scan the massive parquet file ONE time.
            conditions = [
                (pl.col("extracted_pos") >= lower) & (pl.col("extracted_pos") <= upper)
                for _, lower, upper in snp_windows
            ]
            combined_filter = reduce(operator.or_, conditions)
            
            # Execute the query: Filter the massive file and collect ONLY the matching rows into RAM
            try:
                df_filtered = lf.filter(combined_filter).collect()
            except Exception as e:
                print(f"  Error reading {os.path.basename(pq_file)}: {e}")
                continue
            
            if len(df_filtered) == 0:
                print(f"  No SNPs found in the 2Mb windows for Brain_{tissue}.")
                continue
                
            # 4. Split the loaded data by rs_id and save to individual TSV files
            for rs_id, lower, upper in snp_windows:
                # Isolate the data for this specific rs_id
                subset = df_filtered.filter(
                    (pl.col("extracted_pos") >= lower) & (pl.col("extracted_pos") <= upper)
                )
                
                if len(subset) > 0:
                    # Drop the temporary position column we created
                    subset = subset.drop("extracted_pos")
                    
                    output_filename = f"{rs_id}_2mb_Brain_{tissue}_apaQTL.tsv"
                    subset.write_csv(output_filename, separator='\t')
                    print(f"  -> Saved {len(subset)} rows to {output_filename}")

    print("\nExtraction Complete!")

if __name__ == "__main__":
    extract_apaqtl_windows()

Loading target SNPs...

--- Processing Chromosome 1 (2 SNPs) ---
Scanning Brain_Cortex...
  -> Saved 6987 rows to rs11209943_2mb_Brain_Cortex_apaQTL.tsv
  -> Saved 8963 rows to rs4348675_2mb_Brain_Cortex_apaQTL.tsv
Scanning Brain_Hippocampus...
  -> Saved 6869 rows to rs11209943_2mb_Brain_Hippocampus_apaQTL.tsv
  -> Saved 8963 rows to rs4348675_2mb_Brain_Hippocampus_apaQTL.tsv
Scanning Brain_Hypothalamus...
  -> Saved 6987 rows to rs11209943_2mb_Brain_Hypothalamus_apaQTL.tsv
  -> Saved 8963 rows to rs4348675_2mb_Brain_Hypothalamus_apaQTL.tsv
Scanning Brain_Substantia_nigra...
  -> Saved 12995 rows to rs11209943_2mb_Brain_Substantia_nigra_apaQTL.tsv
  -> Saved 14131 rows to rs4348675_2mb_Brain_Substantia_nigra_apaQTL.tsv
Scanning Brain_Frontal_Cortex_BA9...
  -> Saved 6987 rows to rs11209943_2mb_Brain_Frontal_Cortex_BA9_apaQTL.tsv
  -> Saved 8963 rows to rs4348675_2mb_Brain_Frontal_Cortex_BA9_apaQTL.tsv
Scanning Brain_Amygdala...
  -> Saved 6869 rows to rs11209943_2mb_Brain_Amygdala_apa

In [3]:
# Print the number of samples for each tissue (columns of the covariates file)

import glob
import os

# 1. Set the path to the folder containing your covariates files
folder_path = r"D:\Project_CompBio\Summer_2026\QTL_Project\apaQTL and ipaQTL\GTEx_apaQTL\Brain_Tissues\apaQTL_covariates" 

# 2. Create the search pattern
search_pattern = os.path.join(folder_path, "*.v11.apaQTL_covariates.txt")

print("Extracting GTEx Sample Sizes...")

# 3. Loop through every file that matches the pattern
for file_path in glob.glob(search_pattern):
    
    # Extract just the file name from the full path
    file_name = os.path.basename(file_path)
    
    # Extract the Tissue name by removing the suffix
    tissue_name = file_name.replace(".v11.apaQTL_covariates.txt", "")
    
    # 4. Open the file and calculate the sample size
    try:
        with open(file_path, 'rt', encoding='utf-8') as file:
            # Read only the first line
            header = file.readline().strip().split('\t')
            
            # Calculate N (Total columns minus the 'ID' column)
            n_qtl = len(header) - 1
            
            # Print the result neatly
            print(f"{tissue_name:<25} | N = {n_qtl}")
            
    except Exception as e:
        print(f"Error reading {file_name}: {e}")

print("Finished!")

Extracting GTEx Sample Sizes...
Brain_Amygdala            | N = 180
Brain_Anterior_cingulate_cortex_BA24 | N = 233
Brain_Caudate_basal_ganglia | N = 298
Brain_Cerebellar_Hemisphere | N = 276
Brain_Cerebellum          | N = 264
Brain_Cortex              | N = 268
Brain_Frontal_Cortex_BA9  | N = 268
Brain_Hippocampus         | N = 254
Brain_Hypothalamus        | N = 256
Brain_Nucleus_accumbens_basal_ganglia | N = 284
Brain_Putamen_basal_ganglia | N = 253
Brain_Spinal_cord_cervical_c-1 | N = 203
Brain_Substantia_nigra    | N = 183
Finished!


In [2]:
# print the rows where the phenotype_id is ENSG00000184584.13_chr5:139475533-139476454

import pandas as pd

# 1. Define the path to your Parquet file
file_path = r"D:\Project_CompBio\Summer_2026\QTL_Project\apaQTL and ipaQTL\GTEx_apaQTL\GTEx_Analysis_v11_apaQTL_SuSiE\GTEx_Analysis_v11_apaQTL\Brain_Frontal_Cortex_BA9.v11.apaQTLs.SuSiE_summary.parquet"

# Your top colocalization hit
target_event = "ENSG00000109881.17_chr11:27348869-27350439"

try:
    # 2. Load the Parquet file into a DataFrame
    # Note: pandas uses the 'pyarrow' or 'fastparquet' engine under the hood for this
    df = pd.read_parquet(file_path)
    
    # 3. Filter the DataFrame for your specific APA site
    target_df = df[df['phenotype_id'] == target_event]
    
    # 4. Force pandas to display ALL rows and columns without truncating
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000) # Prevents text from wrapping weirdly
    
    # 5. Print the results
    print(f"Found {len(target_df)} rows for {target_event}:\n")
    print(target_df)
    
except Exception as e:
    print(f"ERROR: {e}")

Found 3 rows for ENSG00000109881.17_chr11:27348869-27350439:

                                     phenotype_id             gene_id gene_name         biotype              variant_id       pip        af  cs_id  cs_size
16912  ENSG00000109881.17_chr11:27348869-27350439  ENSG00000109881.17    CCDC34  protein_coding  chr11_27342713_G_A_b38  0.358941  0.283582      1        3
16913  ENSG00000109881.17_chr11:27348869-27350439  ENSG00000109881.17    CCDC34  protein_coding  chr11_27348359_T_C_b38  0.358941  0.283582      1        3
16914  ENSG00000109881.17_chr11:27348869-27350439  ENSG00000109881.17    CCDC34  protein_coding  chr11_27349041_T_C_b38  0.273282  0.285448      1        3
